# Export the papaya cascade for Android

This notebook converts the same YOLOv11m and EfficientNet-B2 checkpoints used by the evaluation notebook. YOLO is exported to NCNN at 832 px, while EfficientNet-B2 is exported to float16 TFLite with float32 RGB input in the 0-255 range.

In [ ]:
# Install export dependencies before loading Ultralytics.
%pip install -q -U "ultralytics>=8.3" "tensorflow==2.19.0" "keras==3.13.2" ncnn pnnx

# If Colab asks for a runtime restart, restart after this cell and continue from the Drive mount cell.


In [ ]:
# Mount Drive and locate the project folder.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import json
import shutil
from pathlib import Path

import numpy as np
import tensorflow as tf
from PIL import Image
from ultralytics import YOLO

PROJECT_NAME = 'Papaya_Leaf_Disease_Classification'
MANUAL_PROJECT_ROOT = None  # Set this to a Path(...) if Drive uses a custom folder name.

def find_project_root():
    if MANUAL_PROJECT_ROOT is not None:
        root = Path(MANUAL_PROJECT_ROOT)
        if not root.is_dir():
            raise FileNotFoundError(f'MANUAL_PROJECT_ROOT does not exist: {root}')
        return root

    drive_roots = [Path('/content/drive/MyDrive'), Path('/content/drive/.shortcut-targets-by-id')]
    candidates = [root / PROJECT_NAME for root in drive_roots if root.is_dir()]
    for drive_root in drive_roots:
        if not drive_root.is_dir():
            continue
        candidates.extend(
            candidate for candidate in drive_root.rglob(PROJECT_NAME)
            if candidate.is_dir() and any((candidate / marker).exists() for marker in ['prepared_data_v1', 'outputs', 'notebooks'])
        )

    unique_candidates = list(dict.fromkeys(candidates))
    # Prefer the copy that actually contains checkpoints if Drive has duplicates/shortcuts.
    for candidate in unique_candidates:
        if candidate.is_dir() and any(candidate.rglob('best.pt')) and any(candidate.rglob('best_model.keras')):
            return candidate
    for candidate in unique_candidates:
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(
        'Project folder was not found on Drive. Check that Drive is mounted, or set MANUAL_PROJECT_ROOT.'
    )

PROJECT_ROOT = find_project_root()
EXPORT_ROOT = PROJECT_ROOT / 'Model' / 'Android_Export'
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('TensorFlow:', tf.__version__)
print('Export directory:', EXPORT_ROOT)


In [ ]:
def normalize_path(path):
    return str(path).lower().replace('_', '').replace('-', '').replace(' ', '')

def find_checkpoint(candidates, preferred_names, required_terms, fallback_names=()):
    for candidate in candidates:
        if candidate.is_file():
            return candidate

    accepted_names = set(preferred_names) | set(fallback_names)
    files = [p for name in accepted_names for p in PROJECT_ROOT.rglob(name) if p.is_file()]
    matches = [p for p in files if all(term in normalize_path(p) for term in required_terms)]
    for name in list(preferred_names) + list(fallback_names):
        same_name = [p for p in matches if p.name == name]
        if same_name:
            return sorted(same_name, key=lambda p: (0 if 'model' in normalize_path(p.parent) else 1, len(str(p))))[0]

    found = sorted(str(p.relative_to(PROJECT_ROOT)) for p in files)
    raise FileNotFoundError(
        f'No checkpoint found under PROJECT_ROOT={PROJECT_ROOT}. '
        f'Required path terms={required_terms}. Files found={found[:30] or "none"}'
    )

YOLO_CKPT = find_checkpoint(
    [
        PROJECT_ROOT / 'Model' / 'YOLOv11' / 'YOLOv11m' / 'weights' / 'best.pt',
        PROJECT_ROOT / 'Model' / 'YOLOv11' / 'yolov11_m' / 'weights' / 'best.pt',
        PROJECT_ROOT / 'outputs' / 'yolo' / 'yolov11_m' / 'weights' / 'best.pt',
        PROJECT_ROOT / 'outputs' / 'yolo' / 'yolov11m' / 'weights' / 'best.pt',
    ],
    ['best.pt'], ['yolo', '11m'], ['last.pt'],
)
CNN_CKPT = find_checkpoint(
    [
        PROJECT_ROOT / 'Model' / 'CNN' / 'EfficientNetB2' / 'best_model.keras',
        PROJECT_ROOT / 'Model' / 'CNN' / 'efficientnet_b2' / 'best_model.keras',
        PROJECT_ROOT / 'outputs' / 'cnn' / 'efficientnet_b2' / 'best_model.keras',
    ],
    ['best_model.keras'], ['efficientnetb2'],
    ['best_stage2.keras', 'last_stage2.keras', 'best_stage1.keras'],
)

print('YOLO checkpoint:', YOLO_CKPT)
print('CNN checkpoint:', CNN_CKPT)


In [ ]:
# Export raw YOLO predictions. NMS and confidence retries run in the Android app.
# Export on Colab's local disk, then copy the two NCNN files to Drive.
LOCAL_YOLO_EXPORT_ROOT = Path('/content/papaya_yolo_ncnn_export')
LOCAL_YOLO_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_YOLO_CKPT = LOCAL_YOLO_EXPORT_ROOT / YOLO_CKPT.name
shutil.copy2(YOLO_CKPT, LOCAL_YOLO_CKPT)

# Clear only this generated local export from a previous run.
LOCAL_NCNN_DIR = LOCAL_YOLO_EXPORT_ROOT / f'{LOCAL_YOLO_CKPT.stem}_ncnn_model'
if LOCAL_NCNN_DIR.exists():
    shutil.rmtree(LOCAL_NCNN_DIR)

yolo = YOLO(str(LOCAL_YOLO_CKPT))
exported_path = Path(yolo.export(
    format='ncnn',
    imgsz=832,
    batch=1,
))
if not exported_path.is_absolute():
    exported_path = (LOCAL_YOLO_EXPORT_ROOT / exported_path).resolve()

if not exported_path.is_dir():
    raise FileNotFoundError(f'NCNN export directory was not created: {exported_path}')

param_files = list(exported_path.rglob('*.param'))
bin_files = list(exported_path.rglob('*.bin'))
if len(param_files) != 1 or len(bin_files) != 1:
    raise RuntimeError(f'Unexpected NCNN export contents: {exported_path}')

YOLO_PARAM_OUT = EXPORT_ROOT / 'yolo11m_papaya.param'
YOLO_BIN_OUT = EXPORT_ROOT / 'yolo11m_papaya.bin'
shutil.copy2(param_files[0], YOLO_PARAM_OUT)
shutil.copy2(bin_files[0], YOLO_BIN_OUT)
print('NCNN files copied to Drive:', YOLO_PARAM_OUT, YOLO_BIN_OUT)


In [ ]:
cnn = tf.keras.models.load_model(CNN_CKPT, compile=False)
print('Keras input:', cnn.input_shape, 'output:', cnn.output_shape)
if tuple(cnn.input_shape[1:]) != (260, 260, 3) or cnn.output_shape[-1] != 5:
    raise ValueError('Expected EfficientNet-B2 input [1,260,260,3] and output [1,5]')

# Prefer the built-in TFLite op set so Android does not need the Flex runtime.
def make_cnn_converter():
    current = tf.lite.TFLiteConverter.from_keras_model(cnn)
    current.optimizations = [tf.lite.Optimize.DEFAULT]
    current.target_spec.supported_types = [tf.float16]
    return current

CNN_USES_SELECT_TF_OPS = False
converter = make_cnn_converter()
try:
    tflite_bytes = converter.convert()
except Exception as exc:
    message = str(exc)
    if 'neither a custom op nor a flex op' not in message:
        raise
    print('Built-in TFLite conversion hit an unsupported TensorFlow op; retrying with Select TF Ops.')
    converter = make_cnn_converter()
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS,
    ]
    tflite_bytes = converter.convert()
    CNN_USES_SELECT_TF_OPS = True

TFLITE_OUT = EXPORT_ROOT / 'efficientnet_b2_papaya_float16.tflite'
TFLITE_OUT.write_bytes(tflite_bytes)
print('TFLite model:', TFLITE_OUT, f'({TFLITE_OUT.stat().st_size / 1e6:.1f} MB)')
print('Requires Select TF Ops/Flex on Android:', CNN_USES_SELECT_TF_OPS)


In [ ]:
# Validate the TFLite signature. Run inference here only when the model uses built-in ops.
interpreter = tf.lite.Interpreter(model_path=str(TFLITE_OUT), num_threads=4)
signatures = interpreter.get_signature_list()
print('TFLite signatures:', signatures)
assert 'serving_default' in signatures

if CNN_USES_SELECT_TF_OPS:
    print('The model contains Select TF Ops. Colab cannot allocate tensors without the Flex delegate; Android must include the Select TF Ops runtime.')
else:
    interpreter.allocate_tensors()
    input_info = interpreter.get_input_details()[0]
    output_info = interpreter.get_output_details()[0]
    print('TFLite input:', input_info['shape'], input_info['dtype'])
    print('TFLite output:', output_info['shape'], output_info['dtype'])
    assert list(input_info['shape']) == [1, 260, 260, 3]
    assert list(output_info['shape']) == [1, 5]
    assert input_info['dtype'] == np.float32

    test_roots = [
        PROJECT_ROOT / 'prepared_data_v1' / 'cnn_dataset' / 'test',
        PROJECT_ROOT / 'Data' / 'prepared_data_v1' / 'cnn_dataset' / 'test',
    ]
    test_images = []
    for root in test_roots:
        if root.exists():
            test_images = [p for p in root.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
            if test_images:
                break

    if test_images:
        sample = np.asarray(Image.open(test_images[0]).convert('RGB').resize((260, 260)), dtype=np.float32)[None, ...]
        keras_probs = cnn.predict(sample, verbose=0)[0]
        interpreter.set_tensor(input_info['index'], sample)
        interpreter.invoke()
        tflite_probs = interpreter.get_tensor(output_info['index'])[0]
        print('Validation image:', test_images[0])
        print('Keras argmax/conf:', int(np.argmax(keras_probs)), float(np.max(keras_probs)))
        print('TFLite argmax/conf:', int(np.argmax(tflite_probs)), float(np.max(tflite_probs)))
        print('Maximum probability difference:', float(np.max(np.abs(keras_probs - tflite_probs))))
    else:
        print('No test image found; tensor shape validation completed.')


In [ ]:
# Keep only deliverables in the package. Build it locally to avoid Drive sync delays and self-inclusion.
LOCAL_PACKAGE_ROOT = Path('/content/papaya_android_package')
if LOCAL_PACKAGE_ROOT.exists():
    shutil.rmtree(LOCAL_PACKAGE_ROOT)
LOCAL_PACKAGE_ROOT.mkdir(parents=True, exist_ok=True)

shutil.copy2(param_files[0], LOCAL_PACKAGE_ROOT / 'yolo11m_papaya.param')
shutil.copy2(bin_files[0], LOCAL_PACKAGE_ROOT / 'yolo11m_papaya.bin')
(LOCAL_PACKAGE_ROOT / 'efficientnet_b2_papaya_float16.tflite').write_bytes(tflite_bytes)

cnn_labels = ['Anthracnose', 'BacterialSpot', 'Curl', 'Healthy', 'RingSpot']
yolo_labels = ['Anthracnose', 'BacterialSpot', 'Curl', 'RingSpot']
cnn_labels_text = '\n'.join(cnn_labels) + '\n'
yolo_labels_text = '\n'.join(yolo_labels) + '\n'
(EXPORT_ROOT / 'papaya_cnn_labels.txt').write_text(cnn_labels_text, encoding='utf-8')
(EXPORT_ROOT / 'papaya_yolo_labels.txt').write_text(yolo_labels_text, encoding='utf-8')
(LOCAL_PACKAGE_ROOT / 'papaya_cnn_labels.txt').write_text(cnn_labels_text, encoding='utf-8')
(LOCAL_PACKAGE_ROOT / 'papaya_yolo_labels.txt').write_text(yolo_labels_text, encoding='utf-8')

config = {
    'yolo_input_size': 832,
    'cnn_input_size': 260,
    'yolo_classes': yolo_labels,
    'cnn_classes': cnn_labels,
    'cnn_requires_select_tf_ops': bool(CNN_USES_SELECT_TF_OPS),
    'initial_yolo_confidence': 0.50,
    'retry_yolo_confidences': [0.45, 0.40, 0.35],
    'yolo_nms_iou': 0.70,
    'full_image_cnn_disease_min_probability': 0.50,
    'bbox_expand_ratio_each_side': 0.10,
    'image_vote': 'sum(cnn_confidence * yolo_confidence) by predicted CNN class',
}
config_text = json.dumps(config, indent=2)
(EXPORT_ROOT / 'papaya_android_config.json').write_text(config_text, encoding='utf-8')
(LOCAL_PACKAGE_ROOT / 'papaya_android_config.json').write_text(config_text, encoding='utf-8')

LOCAL_ARCHIVE = Path(shutil.make_archive('/content/papaya_android_models', 'zip', LOCAL_PACKAGE_ROOT))
ARCHIVE_OUT = EXPORT_ROOT / LOCAL_ARCHIVE.name
shutil.copy2(LOCAL_ARCHIVE, ARCHIVE_OUT)
print('Package on Drive:', ARCHIVE_OUT)
print('Package size:', f'{LOCAL_ARCHIVE.stat().st_size / 1e6:.1f} MB')
print('Package files:', sorted(p.name for p in LOCAL_PACKAGE_ROOT.iterdir()))
print('Copy these three model files to AndroidApp/app/src/main/assets:')
print('yolo11m_papaya.param')
print('yolo11m_papaya.bin')
print('efficientnet_b2_papaya_float16.tflite')


In [ ]:
# Download the local archive directly; this avoids waiting for Google Drive synchronization.
from google.colab import files
files.download(str(LOCAL_ARCHIVE))
